In [ ]:

# -*- coding: utf-8 -*-
"""
Send an email per subfolder under a user-provided root.
- To: subfolder name (assumed to be an email address)
- Subject/Body: standard text (customize below)
- Attach: all files in that subfolder
"""

import os
import sys
from datetime import datetime

try:
    import win32com.client as win32
except ImportError:
    print("Missing dependency: pywin32. Install with: pip install pywin32")
    sys.exit(1)

def is_valid_email(candidate: str) -> bool:
    # Very light validation; replace with stricter regex if needed
    return "@" in candidate and "." in candidate and " " not in candidate

def main():
    root = input("Enter full path of the parent folder (e.g., C:\\FDOT\\Output): ").strip()
    if not os.path.isdir(root):
        print(f"Folder not found: {root}")
        return

    outlook = win32.Dispatch("Outlook.Application")
    mapi = outlook.GetNamespace("MAPI")

    # Customize these:
    SUBJECT_TEMPLATE = "Billing Invoices"
    BODY_TEMPLATE = (
        "Attached please find your billing for this month. This inbox is not monitored so please contact the biller noted below if you have any questions.  "
        "Thank you."
        "For our Jacksonville, Orlando, Tallahassee, Panama City, or Pensacola Branches, contact "
        "Stephanie Mata:	smata@acmebarricades.com"
        "For our Miami, West Palm Beach, Tampa, or Ft Myers Branches, contact"
        "Laura Pittman:	lpittman@acmebarricades.com"
        "For our Alabama, Tennessee, Barrier Wall, Guardrail, Perm Signs, or Striping Divisions, contact"
        "Megan Buckley-Finley:	mbuckleyfinley@acmebarricades.com "
        "For all other questions, please contact Kewanna Groman, Billing Team Lead at kgroman@acmebarricades.com.")

    # Iterate immediate subfolders
    for entry in os.scandir(root):
        if not entry.is_dir():
            continue

        folder_name = entry.name
        recipient = folder_name  # assumes email address

        if not is_valid_email(recipient):
            print(f"Skipping folder '{folder_name}' (not a valid email): {recipient}")
            continue

        # Collect files to attach from this subfolder
        file_paths = []
        for file_entry in os.scandir(entry.path):
            if file_entry.is_file():
                file_paths.append(file_entry.path)

        # Create and send email
        mail = outlook.CreateItem(0)  # 0 = olMailItem
        mail.To = recipient
        mail.Subject = SUBJECT_TEMPLATE.format(folder=folder_name)
        mail.Body = BODY_TEMPLATE.format(folder=folder_name, date=datetime.now().strftime("%Y-%m-%d"))

        # Attach files
        for fp in file_paths:
            mail.Attachments.Add(fp)

        # Optional: display before sending (for review)
        # mail.Display()  # comment this out to auto-send
        mail.Send()
        print(f"Sent email to {recipient} with {len(file_paths)} attachment(s).")

if __name__ == "__main__":
    main()
